In [1]:
import os 
os.chdir('../../../../')
os.environ["DPM_TQDM"] = "False"
os.environ["CUDA_VISIBLE_DEVICES"]="1"

!nvidia-smi

Thu Aug 14 20:36:57 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        Off |   00000000:19:00.0 Off |                  Off |
| 79%   82C    P2            359W /  450W |   16535MiB /  24564MiB |     98%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# -*- coding: utf-8 -*-
import os
import math
import numpy as np
from easydict import EasyDict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

# ===============================
# Config
# ===============================
config = EasyDict()
config.backbone      = 'DiT'
config.train_pt_dir  = 'samplings/dit/train_4.0/dit_train_4.0_1'
config.valid_pt_dir  = 'samplings/dit/eval1000_4.0/dit_eval1000_4.0_0'
config.batch_size    = 10
config.CFG           = 4.0
config.epochs        = 10
config.val_every     = 100
config.log_dir       = "logs/CFG4.0/0814-12:ema-third"

# LR & Scheduler (constant LR)
config.base_lr       = 1e-3
config.total_steps   = 10000
config.warmup_steps  = 0           # not used for constant schedule
config.min_lr_ratio  = 1.0         # not used for constant schedule

# EMA
config.ema_decay         = 0.99
config.ema_use_for_eval  = True
config.ema_start_step    = 100       # set >0 for burn-in (e.g., 500)
config.ema_rampup_steps  = 0       # set >0 to ramp decay to target

os.makedirs(config.log_dir, exist_ok=True)


# ===============================
# Model (frozen backbone)
# ===============================
from backbones.dit import DiT
from utils.inception import FIDInception

if config.backbone == 'DiT':
    model = DiT(trainable=True)
    model.set_freeze()
device = model.device
print(model)
inception = FIDInception().to(device)


# ===============================
# Dataset / Dataloader
# ===============================
from datasets.pt_dataset import PtDataset

train_dataset = PtDataset(config.train_pt_dir)
valid_dataset = PtDataset(config.valid_pt_dir)
print('len(train_dataset) :', len(train_dataset), 'len(valid_dataset) :', len(valid_dataset))

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)

valid_loader = DataLoader(valid_dataset, batch_size=config.batch_size, shuffle=False)
print('dataloaders ready')


# ===============================
# Solver / Optimizer / Scheduler
# ===============================
from solvers.dual.dynamic.gdual_solver_log_deltaL_only_3rd import GDual_Solver
from solvers.transforms.loglinear_transform_general import LogLinearTransform
from solvers.param_extractors.gap_extractor import GAP_Extractor

noise_schedule = model.get_noise_schedule()
extractor = GAP_Extractor(input_shape=(4, 32, 32))
transform = LogLinearTransform(gamma_push=True, gamma_max=3, kappa_max=3, kappa_disable=False)
solver = GDual_Solver(
    noise_schedule,
    steps=5,
    transform=transform,
    param_extractor=extractor,
    skip_type="time_uniform",
    order=2,                 # can be 1, 2, or 3 (your 3rd-order impl)
    time_learning=True,
    train_mode=True
).to(device)

optimizer = torch.optim.AdamW(solver.parameters(), lr=config.base_lr)
print('solver/optimizer')

# constant LR
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda step: 1.0)


# ===============================
# EMA utils
# ===============================
class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.999, device=None):
        self.decay = float(decay)
        self.shadow = {}
        self.backup = {}
        self.device = device
        self.register(model)

    @torch.no_grad()
    def register(self, model: nn.Module):
        self.shadow.clear()
        for name, param in model.named_parameters():
            if param.requires_grad:
                data = param.data
                if self.device is not None:
                    data = data.to(self.device)
                self.shadow[name] = data.clone()

    @torch.no_grad()
    def update(self, model: nn.Module):
        for name, param in model.named_parameters():
            if not param.requires_grad:
                continue
            assert name in self.shadow
            ema_param = self.shadow[name]
            # in-place EMA update
            ema_param.mul_(self.decay).add_(param.data, alpha=1.0 - self.decay)

    def store(self, model: nn.Module):
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()

    @torch.no_grad()
    def copy_to(self, model: nn.Module):
        for name, param in model.named_parameters():
            if param.requires_grad:
                param.data.copy_(self.shadow[name])

    @torch.no_grad()
    def restore(self, model: nn.Module):
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.backup:
                param.data.copy_(self.backup[name])
        self.backup = {}

    def state_dict(self):
        return {
            "decay": self.decay,
            "shadow": {k: v.cpu() for k, v in self.shadow.items()}
        }

    def load_state_dict(self, state):
        self.decay = float(state["decay"])
        self.shadow = {k: v.clone() for k, v in state["shadow"].items()}


ema = EMA(solver, decay=config.ema_decay)  # start tracking immediately


# ===============================
# Utils
# ===============================
def abort_if_bad(tag, value, step=None):
    v = float(value.detach().cpu()) if isinstance(value, torch.Tensor) else float(value)
    if (not math.isfinite(v)) or (v >= 100.0):
        msg = f"[EARLY-STOP] {tag} loss={v:.6f}" + (f" @ step {step}" if step is not None else "")
        print(msg, flush=True)
        raise RuntimeError(msg)

def save_checkpoint(global_step, save_dir, solver, valid_loss, ema_obj: EMA = None):
    ckpt = {
        "global_step": int(global_step),
        "solver_state_dict": solver.state_dict(),
        "valid_loss": float(valid_loss),
        "config": dict(config),
    }
    if ema_obj is not None:
        ckpt["ema_state_dict"] = ema_obj.state_dict()
    os.makedirs(save_dir, exist_ok=True)
    step_path = os.path.join(save_dir, f"step_{global_step:08d}.pt")
    torch.save(ckpt, step_path)
    return step_path

@torch.no_grad()
def get_valid_loss(device, solver, ema_obj: EMA = None, use_ema: bool = True):
    solver.eval()
    # swap to EMA weights if requested
    swapped = False
    if ema_obj is not None and use_ema:
        ema_obj.store(solver)
        ema_obj.copy_to(solver)
        swapped = True

    psnr_losses = []
    inception_losses = []
    pbar = tqdm(valid_loader, leave=False)
    for bi, batch in enumerate(pbar):
        noises = batch['noise'].to(device, non_blocking=True)
        conds  = batch['cond']
        targets= batch['sample'].to(device, non_blocking=True)
        target_features= batch['inception_feature'][:, 0].to(device, non_blocking=True)

        model_fn = model.get_model_fn(noise_schedule, pos_conds=conds, guidance_scale=config.CFG)
        pred = solver.sample(noises, model_fn)
        loss = psnr_loss = torch.log(F.mse_loss(pred, targets) + 1e-8)

        pred = model.decode_vae(pred, raw_output=True)
        pred = inception(pred)
        inception_loss = F.mse_loss(pred, target_features)

        abort_if_bad("valid(batch)", loss)

        psnr_losses.append(psnr_loss.item())
        inception_losses.append(inception_loss.item())
        pbar.set_postfix({'val_loss': loss.item()})

    # restore original weights if swapped
    if swapped:
        ema_obj.restore(solver)

    val_psnr_mean = float(np.mean(psnr_losses))
    val_inception_mean = float(np.mean(inception_losses))
    abort_if_bad("valid(mean)", val_inception_mean)
    return val_psnr_mean, val_inception_mean


def do_train_loop(device, epoch, writer, solver, optimizer, scheduler, ema_obj: EMA, global_step_start=0):
    solver.train()
    pbar = tqdm(train_loader)
    losses = []
    global_step = global_step_start
    base_decay = config.ema_decay

    for step, batch in enumerate(pbar):
        if global_step >= config.total_steps:
            break

        # periodic validation
        if global_step > 0 and global_step % config.val_every == 0:
            use_ema_now = (global_step >= config.ema_start_step) and config.ema_use_for_eval
            val_psnr_mean, val_inception_mean = get_valid_loss(device, solver, ema_obj, use_ema=use_ema_now)
            print(f'step : {global_step} valid_psnr_loss : {val_psnr_mean:.6f}')
            print(f'step : {global_step} valid_inception_loss : {val_inception_mean:.6f}')
            writer.add_scalar("valid/psnr_loss", val_psnr_mean, global_step)
            writer.add_scalar("valid/inception_loss", val_inception_mean, global_step)
            save_checkpoint(global_step, config.log_dir, solver, val_inception_mean, ema_obj)

        optimizer.zero_grad(set_to_none=True)
        noises = batch['noise'].to(device, non_blocking=True)
        conds  = batch['cond']
        targets= batch['sample'].to(device, non_blocking=True)
        target_features = batch['inception_feature'][:, 0].to(device, non_blocking=True)

        model_fn = model.get_model_fn(noise_schedule, pos_conds=conds, guidance_scale=config.CFG)

        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            pred = solver.sample(noises, model_fn)
            psnr_loss = torch.log(F.mse_loss(pred, targets) + 1e-8)
            pred = model.decode_vae(pred, raw_output=True)
            pred = inception(pred)
            loss = inception_loss = F.mse_loss(pred, target_features)
            cosine_loss = torch.mean(F.cosine_similarity(pred, target_features))

        abort_if_bad("train", loss, global_step)

        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(solver.parameters(), 1.0)
        if not torch.isfinite(grad_norm):
            print(f"[SKIP-STEP] non-finite grad_norm={float(grad_norm):.4e}", flush=True)
            optimizer.zero_grad(set_to_none=True)
            continue

        optimizer.step()
        scheduler.step()

        # EMA update (with optional burn-in & ramp-up)
        use_ema_update = (global_step >= config.ema_start_step)
        if use_ema_update:
            if config.ema_rampup_steps > 0:
                t = (global_step - config.ema_start_step) / max(1, config.ema_rampup_steps)
                t = float(min(1.0, max(0.0, t)))
                ema_obj.decay = 1.0 - (1.0 - base_decay) * t
            else:
                ema_obj.decay = base_decay
            ema_obj.update(solver)

        lr_now = optimizer.param_groups[0]["lr"]
        writer.add_scalar("train/lr", lr_now, global_step)
        writer.add_scalar("train/psnr_loss", psnr_loss.item(), global_step)
        writer.add_scalar("train/inception_loss", inception_loss.item(), global_step)
        writer.add_scalar("train/cosine_loss", cosine_loss.item(), global_step)
        if use_ema_update:
            writer.add_scalar("train/ema_decay", ema_obj.decay, global_step)

        losses.append(loss.item())
        pbar.set_postfix({'loss': loss.item(), 'lr': lr_now})
        global_step += 1

    return float(np.mean(losses)) if losses else 0.0, global_step


/home/scpark/miniconda3/envs/rbf/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading pipeline components...:   0%|          | 0/3 [00:00<?, ?it/s]An error occurred while trying to fetch /data/huggingface/DiT-XL-2-256/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /data/huggingface/DiT-XL-2-256/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
An error occurred while trying to fetch /data/huggingface/DiT-XL-2-256/transformer: Error no file named diffusion_pytorch_model.safetensors found in directory /data/huggingface/DiT-XL-2-256/transformer.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Loading pipeline components...: 100%|██████████| 3/3 [00:00<00:00,  4.54it/s]
Expected types for

len(train_dataset) : 10000 len(valid_dataset) : 1000
dataloaders ready
solver/optimizer


In [ ]:
# ===============================
# Train
# ===============================
def main():
    writer = SummaryWriter(config.log_dir)
    print('tensorboard:', config.log_dir)

    global_step = 0
    for epoch in range(config.epochs):
        if global_step >= config.total_steps:
            break
        mean_loss, global_step = do_train_loop(
            device, epoch, writer, solver, optimizer, scheduler, ema, global_step_start=global_step
        )
        print(f'[epoch {epoch}] mean_train_loss={mean_loss:.6f}, global_step={global_step}')

    # final validation & checkpoint (EMA gated as during training)
    use_ema_now = (global_step >= config.ema_start_step) and config.ema_use_for_eval
    val_psnr_mean, val_inception_mean = get_valid_loss(device, solver, ema, use_ema=use_ema_now)
    save_checkpoint(global_step, config.log_dir, solver, val_inception_mean, ema)
    writer.add_scalar("valid/loss_final", val_inception_mean, global_step)
    writer.close()
    print('done')


if __name__ == "__main__":
    torch.backends.cudnn.benchmark = True
    main()


tensorboard: logs/CFG4.0/0814-12:ema-third


 10%|█         | 100/1000 [01:55<17:23,  1.16s/it, loss=0.0421, lr=0.001]

step : 100 valid_psnr_loss : 3.610899
step : 100 valid_inception_loss : 0.203334


 20%|██        | 200/1000 [04:20<14:50,  1.11s/it, loss=0.0358, lr=0.001]  

step : 200 valid_psnr_loss : -1.109561
step : 200 valid_inception_loss : 0.048166


 24%|██▍       | 240/1000 [05:40<14:58,  1.18s/it, loss=0.0357, lr=0.001]  